In [1]:
# Install Unsloth natively via their official pre-compiled GitHub wheel
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install essential dependencies without dependency-resolution lockups
!pip install --no-deps xformers trl peft accelerate bitsandbytes

# Install utility libraries for dataset management and evaluation
!pip install kagglehub datasets transformers evaluate rouge_score bleu bert_score

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-2yvcxd3m/unsloth_2ece09209c9047cb889dfb42e17272a1
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-2yvcxd3m/unsloth_2ece09209c9047cb889dfb42e17272a1
  Resolved https://github.com/unslothai/unsloth.git to commit 983c1f0616b65e3c26a3dc3eeee389e8c9a28ae7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB

In [2]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback
from datasets import Dataset
import json
import os

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
def load_and_format_dataset(file_path):
    formatted_data = []
    skipped_count = 0

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Input file not found at: {file_path}")

    with open(file_path, 'r', encoding='utf-8') as f:
        first_char = f.read(1)
        f.seek(0)

        if first_char == '[':
            records = json.load(f)
        else:
            records = []
            for line in f:
                cleaned_line = line.strip()
                if not cleaned_line: continue
                try:
                    records.append(json.loads(cleaned_line))
                except json.JSONDecodeError:
                    skipped_count += 1
                    continue

    for record in records:
        jd = record.get("job_description", "")
        keyword_profile = record.get("keyword_profile", {})

        if not jd or not keyword_profile:
            continue

        target_output = json.dumps(keyword_profile, ensure_ascii=False)

        messages = [
            {
                "role": "system",
                "content": "You are a professional HR data extraction assistant. Analyze the provided Job Description and extract the key terms into a structured JSON profile containing 'primary', 'secondary', and 'tertiary' keyword arrays. Output ONLY the raw JSON block without formatting wrappers or commentary."
            },
            {
                "role": "user",
                "content": f"Extract the keyword profile from this job description:\n\n{jd}"
            },
            {
                "role": "assistant",
                "content": target_output
            }
        ]
        formatted_data.append({"messages": messages})

    print(f"Successfully loaded {len(formatted_data)} records. (Skipped: {skipped_count} rows)")
    return Dataset.from_list(formatted_data)

def apply_formatting_template(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

In [4]:
INPUT_FILE = "/kaggle/input/datasets/adhamashraf202200953/techncial-generated-job-descriptions/era_match_keywords.jsonl" 
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct"  
OUTPUT_DIR = "/kaggle/working/outputs/jd_to_keywords_model"
CHECKPOINT_DIR = "/kaggle/working/outputs/checkpoints"       
MAX_SEQ_LENGTH = 4048
LOAD_IN_4BIT = True

raw_dataset = load_and_format_dataset(INPUT_FILE)
dataset_split = raw_dataset.train_test_split(test_size=0.05, seed=3407)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

tokenizer = get_chat_template(tokenizer, chat_template = "qwen-2.5")
tokenizer.eos_token = "<|im_end|>"
tokenizer.pad_token = tokenizer.eos_token

train_dataset = train_dataset.map(apply_formatting_template, batched = True)
eval_dataset = eval_dataset.map(apply_formatting_template, batched = True)

train_dataset = train_dataset.remove_columns(["messages"])
eval_dataset = eval_dataset.remove_columns(["messages"])

Successfully loaded 5014 records. (Skipped: 0 rows)
==((====))==  Unsloth 2026.6.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[unsloth_zoo.log|WARNING]Unsloth: Could not apply RoPE scaling 'default' from config (KeyError: 'default'); falling back to unscaled RoPE. Long-context generation may degrade.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Map:   0%|          | 0/4763 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

In [ ]:
args = SFTConfig(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    per_device_eval_batch_size = 1,
    eval_accumulation_steps = 1,
    warmup_steps = 10,
    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = True,
    bf16 = False,
    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 100,
    save_strategy = "steps",
    save_steps = 100,
    save_total_limit = 2,
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    gradient_checkpointing = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    optim = "adamw_8bit",
    weight_decay = 0.01,
    seed = 3407,
    output_dir = CHECKPOINT_DIR,    
    remove_unused_columns = True,
    max_length = MAX_SEQ_LENGTH,
    dataset_text_field = "text",
)

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    processing_class = tokenizer,
    args = args,
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 2)]
)

print("\n--- Starting Training Loop ---")
# trainer_stats = trainer.train(resume_from_checkpoint = True)
import os
resume = True if os.path.exists(CHECKPOINT_DIR) and any("checkpoint-" in d for d in os.listdir(CHECKPOINT_DIR)) else None
trainer_stats = trainer.train(resume_from_checkpoint = resume)

In [7]:
import os
from peft import PeftModel

checkpoint_path = "/kaggle/input/notebooks/adhamashraf202200953/fine-tunning-jd-keywordextraction-qwen-2-5-7b/outputs/checkpoints/checkpoint-596"
print(f"Loading checkpoint from: {checkpoint_path}")
model = PeftModel.from_pretrained(model, checkpoint_path)
OUTPUT_DIR = "/kaggle/working/outputs/jd_to_keywords_model"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Loading checkpoint from: /kaggle/input/notebooks/adhamashraf202200953/fine-tunning-jd-keywordextraction-qwen-2-5-7b/outputs/checkpoints/checkpoint-596


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.o_proj.

('/kaggle/working/outputs/jd_to_keywords_model/tokenizer_config.json',
 '/kaggle/working/outputs/jd_to_keywords_model/chat_template.jinja',
 '/kaggle/working/outputs/jd_to_keywords_model/tokenizer.json')

In [8]:
!zip -r "/kaggle/working/keywords_model.zip" "/kaggle/working/outputs/jd_to_keywords_model"

  adding: kaggle/working/outputs/jd_to_keywords_model/ (stored 0%)
  adding: kaggle/working/outputs/jd_to_keywords_model/adapter_config.json (deflated 58%)
  adding: kaggle/working/outputs/jd_to_keywords_model/chat_template.jinja (deflated 71%)
  adding: kaggle/working/outputs/jd_to_keywords_model/tokenizer_config.json (deflated 89%)
  adding: kaggle/working/outputs/jd_to_keywords_model/tokenizer.json (deflated 81%)
  adding: kaggle/working/outputs/jd_to_keywords_model/README.md (deflated 66%)
  adding: kaggle/working/outputs/jd_to_keywords_model/adapter_model.safetensors (deflated 61%)


In [9]:
print("\n--- Running Validation Inference ---")
FastLanguageModel.for_inference(model)

sample_jd = """
We are looking for a Senior DevOps Engineer with 5+ years of experience.
Must have strong expertise in Kubernetes cluster administration, Terraform for AWS Infrastructure,
and building Gitlab CI/CD pipelines. Experience with Prometheus and Grafana is a plus.
"""

test_messages = [
    {
        "role": "system",
        "content": "You are a professional HR data extraction assistant. Analyze the provided Job Description and extract the key terms into a structured JSON profile containing 'primary', 'secondary', and 'tertiary' keyword arrays. Output ONLY the raw JSON block without formatting wrappers or commentary."
    },
    {
        "role": "user",
        "content": f"Extract the keyword profile from this job description:\n\n{sample_jd}"
    }
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,
    top_p = 0.9,
    pad_token_id = tokenizer.eos_token_id
)

generated_output = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print("Extracted Profile Target Output:\n", generated_output)
print("="*60)


--- Running Validation Inference ---


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Extracted Profile Target Output:
 {
    "primary": ["Senior DevOps Engineer", "5+ years of experience", "Kubernetes cluster administration", "Terraform for AWS Infrastructure", "Gitlab CI/CD pipelines"],
    "secondary": ["DevOps Engineer", "expertise", "AWS Infrastructure", "building", "Prometheus", "Grafana"],
    "tertiary": ["experience", "pipelines"]
}
